<a href="https://colab.research.google.com/github/Kevaru111/travel-packing-assistant/blob/main/travel_packing_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧳 AI Travel Packing Assistant

**Personalizuotas kelionės daiktų sąrašo generatorius naudojant generatyvinį dirbtinį intelektą.**

Sistema priima kelionės informaciją ir, naudodama **Gemini API**, sugeneruoja struktūrizuotą, personalizuotą daiktų sąrašą.

**Procesas:**

`Kelionės duomenys → Validacija → Promptas → Gemini API → JSON atsakymas → Rezultatai → Vizualizacija`

### Sistemos įvestis
- kelionės vieta;
- trukmė dienomis;
- sezonas;
- kelionės tipas;
- planuojamos veiklos;
- bagažo tipas;
- papildomi pageidavimai.

### Sistemos išvestis
- kelionės santrauka;
- daiktai pagal kategorijas;
- kiekvieno daikto prioritetas;
- trumpas rekomendacijos paaiškinimas;
- bagažo optimizavimo rekomendacijos.


## 🔗 SETUP: Gemini API paruošimas

1. Google AI Studio susikurkite Gemini API raktą.
2. Google Colab kairėje atidarykite **Secrets**.
3. Sukurkite secret pavadinimu **`GOOGLE_API_KEY`**.
4. Įklijuokite API raktą ir suteikite notebook'ui prieigą prie šio secret.

Toliau įdiegiamas `google-genai` paketas ir sukuriamas Gemini klientas.

> API raktas nėra įrašomas į kodą ir neturi būti keliamas į GitHub.


In [ ]:
#@title 🔗 SETUP: Install package and configure Gemini API
!pip install -q -U google-genai

from google import genai
from google.colab import userdata
import json

api_key = userdata.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY nerastas. Colab -> Secrets pridėkite secret pavadinimu GOOGLE_API_KEY."
    )

client = genai.Client(api_key=api_key)
MODEL = "gemini-3.8-flash"

print("✅ Gemini API paruoštas")
print("Modelis:", MODEL)


## 🧳 STAGE 1: Kelionės duomenų įvestis

Šiame etape vartotojas pateikia kelionės informaciją. Colab laukus galima pakeisti ir paleisti langelį iš naujo.

Šie duomenys yra sistemos **X (input)**.


In [ ]:
#@title 🧳 STAGE 1: Travel Input

destination = "Rome, Italy" #@param {type:"string"}
duration_days = 5 #@param {type:"integer"}
season = "summer" #@param ["spring", "summer", "autumn", "winter"]
trip_type = "city_trip" #@param ["city_trip", "business_trip", "beach_trip", "hiking_trip", "winter_trip"]
activities = "walking around the city, visiting museums, restaurants" #@param {type:"string"}
luggage_type = "carry_on" #@param ["carry_on", "checked_luggage", "backpack"]
additional_preferences = "Pack light and avoid unnecessary items." #@param {type:"string"}

trip_data = {
    "destination": destination,
    "duration_days": duration_days,
    "season": season,
    "trip_type": trip_type,
    "activities": [a.strip() for a in activities.split(",") if a.strip()],
    "luggage_type": luggage_type,
    "additional_preferences": additional_preferences.strip()
}

print("✅ Įvesties duomenys:")
print(json.dumps(trip_data, ensure_ascii=False, indent=2))


## ✅ STAGE 2: Įvesties validacija

Prieš siunčiant informaciją į Gemini API patikrinama, ar užpildyti pagrindiniai laukai ir ar kelionės trukmė yra teigiama.


In [ ]:
#@title ✅ STAGE 2: Validate Input

def validate_trip_data(data):
    errors = []

    if not data["destination"]:
        errors.append("Nenurodyta kelionės vieta.")
    if not isinstance(data["duration_days"], int) or data["duration_days"] <= 0:
        errors.append("Kelionės trukmė turi būti teigiamas sveikasis skaičius.")
    if not data["season"]:
        errors.append("Nenurodytas sezonas.")
    if not data["trip_type"]:
        errors.append("Nenurodytas kelionės tipas.")
    if not data["activities"]:
        errors.append("Nenurodyta bent viena veikla.")
    if not data["luggage_type"]:
        errors.append("Nenurodytas bagažo tipas.")

    return errors

validation_errors = validate_trip_data(trip_data)

if validation_errors:
    print("❌ Rastos įvesties klaidos:")
    for error in validation_errors:
        print("-", error)
    raise ValueError("Pataisykite įvesties duomenis ir paleiskite langelį dar kartą.")

print("✅ Įvesties duomenys validūs")


## 📝 STAGE 3: Prompt'o formavimas

Validuoti kelionės duomenys paverčiami aiškia užklausa Gemini modeliui. Prompt'e apibrėžiama AI asistento rolė, įvestis ir reikalaujama JSON išvesties struktūra.


In [ ]:
#@title 📝 STAGE 3: Build Prompt

def build_prompt(data):
    return f"""
You are an AI Travel Packing Assistant.

Generate a personalized travel packing list using only the travel data below.

Requirements:
1. Analyze destination, trip duration, season, trip type, activities,
   luggage type and additional preferences.
2. Divide items into these categories:
   - documents
   - clothing
   - electronics
   - hygiene
   - additional
3. For every item return:
   - item
   - priority: essential, high, medium, or low
   - a short reason
4. Add 2-4 baggage optimization recommendations.
5. Do not claim access to live weather, flights, maps or other external services.
6. Keep the recommendations practical and appropriate for the luggage type.

Return this JSON structure:
{{
  "trip_summary": "short summary",
  "packing_list": {{
    "documents": [
      {{
        "item": "...",
        "priority": "essential",
        "reason": "..."
      }}
    ],
    "clothing": [],
    "electronics": [],
    "hygiene": [],
    "additional": []
  }},
  "baggage_optimization": ["..."]
}}

TRAVEL DATA:
{json.dumps(data, ensure_ascii=False, indent=2)}
""".strip()

prompt = build_prompt(trip_data)

print("✅ Promptas sukurtas")
print("\n--- PROMPT PREVIEW ---\n")
print(prompt)


## 🤖 STAGE 4: Užklausa Gemini API

Sukurtas promptas siunčiamas į Gemini API. Prašoma grąžinti **JSON**, kad rezultatą būtų galima lengvai apdoroti Python kode.


In [ ]:
#@title 🤖 STAGE 4: Generate Packing List with Gemini

import time
from google.genai import types

packing_data = None
last_error = None

for attempt in range(1, 4):
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.4
            ),
        )

        packing_data = json.loads(response.text)
        break

    except Exception as e:
        last_error = e
        print(f"Bandymas {attempt} nepavyko: {str(e)[:120]}")
        if attempt < 3:
            time.sleep(3 * attempt)

if packing_data is None:
    raise RuntimeError(f"Nepavyko gauti Gemini atsakymo: {last_error}")

print("✅ Gemini atsakymas gautas")
print(json.dumps(packing_data, ensure_ascii=False, indent=2))


## 📋 STAGE 5: Rezultatų pateikimas

Gemini JSON atsakymas paverčiamas žmogui lengvai skaitomu packing list. Tai yra galutinė sistemos **y (output)**.


In [ ]:
#@title 📋 STAGE 5: Display Results

from IPython.display import display, Markdown

display(Markdown(f"## 🧳 {packing_data.get('trip_summary', 'Packing list')}"))

packing_list = packing_data.get("packing_list", {})

category_names = {
    "documents": "📄 Dokumentai",
    "clothing": "👕 Drabužiai",
    "electronics": "🔌 Elektronika",
    "hygiene": "🧴 Higiena",
    "additional": "🎒 Papildomi daiktai",
}

for category, title in category_names.items():
    items = packing_list.get(category, [])
    if not items:
        continue

    lines = [f"### {title}"]
    for item in items:
        name = item.get("item", "Nenurodyta")
        priority = item.get("priority", "n/a")
        reason = item.get("reason", "")
        lines.append(f"- **{name}** — `{priority}`  \n  {reason}")

    display(Markdown("\n".join(lines)))

recommendations = packing_data.get("baggage_optimization", [])
if recommendations:
    display(Markdown(
        "### 💡 Bagažo optimizavimas\n" +
        "\n".join(f"{i}. {rec}" for i, rec in enumerate(recommendations, 1))
    ))


## 📊 STAGE 6: Paprasta rezultatų vizualizacija

Grafikas parodo, kiek daiktų Gemini sugeneravo kiekvienoje kategorijoje. Vizualizacija nėra būtina sistemos veikimui, tačiau aiškiai parodo rezultato struktūrą.


In [ ]:
#@title 📊 STAGE 6: Visualize Packing List

import matplotlib.pyplot as plt

category_counts = {
    category: len(packing_list.get(category, []))
    for category in category_names
}

labels = [category_names[c].split(" ", 1)[1] for c in category_counts]
values = list(category_counts.values())

plt.figure(figsize=(8, 4))
plt.bar(labels, values)
plt.title("Daiktų skaičius pagal kategorijas")
plt.ylabel("Daiktų skaičius")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 💾 STAGE 7: Rezultatų išsaugojimas

Paskutiniame etape išsaugomi pradiniai kelionės duomenys ir Gemini sugeneruotas rezultatas. Taip aiškiai matomas pilnas End-to-End kelias nuo **X** iki **y**.


In [ ]:
#@title 💾 STAGE 7: Save Input and Output

with open("trip_input.json", "w", encoding="utf-8") as f:
    json.dump(trip_data, f, ensure_ascii=False, indent=2)

with open("packing_list_output.json", "w", encoding="utf-8") as f:
    json.dump(packing_data, f, ensure_ascii=False, indent=2)

print("✅ Išsaugoti failai:")
print("- trip_input.json")
print("- packing_list_output.json")


## ✅ Išvada

Prototipas demonstruoja visą DI agento veikimo procesą:

1. vartotojas pateikia kelionės informaciją;
2. sistema validuoja įvestį;
3. suformuojamas promptas;
4. promptas siunčiamas į Gemini API;
5. Gemini sugeneruoja struktūrizuotą JSON atsakymą;
6. atsakymas pateikiamas kaip personalizuotas packing list;
7. rezultatai vizualizuojami;
8. įvestis ir išvestis išsaugomos JSON failuose.


